In [1]:
from datetime import date
from pathlib import Path

from data_engineering.adsb.read_adsb import read_icaos_from_adsb
from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from evaluation.flights import evaluation
from flights.flights_match_adsb import get_matching_icaos_in_flights
from flights_ai_model.features import get_icaos_in_features, read_icaos_from_features
from flights_ai_model.inference import inference
from flights_ai_model.train import train
from flights_ai_model.training.training_days import sequence_training_days
import polars as pl
from flights_ai_model.validation import flights_df_comparison

In [20]:
val_date = date(2026,3,1)
sfdps_flights = get_bts_flights_for_day(val_date)
icaos = sfdps_flights.get_column("icao").to_list()
df_adsb = read_icaos_from_adsb(val_date, icaos)
df_gold = get_matching_icaos_in_flights(sfdps_flights, use_readsb_parquet=True, target_date=val_date)
icaos = list(set(df_gold.get_column("icao")))

226 overlapping flights found for 2026-03-01


In [ ]:
# from data_engineering.utils import OUTPUT_DIR
# df_gold.write_parquet(OUTPUT_DIR / "tmp/df_gold.parquet")

In [ ]:
# from data_engineering.utils import OUTPUT_DIR
# df_gold = pl.read_parquet(OUTPUT_DIR / "tmp/df_gold.parquet")
# icaos = list(set(df_gold.get_column("icao")))

In [21]:
from datetime import date
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from flights_algorithm.main import run_main_2
df_test = run_main_2(val_date, icaos=icaos)
# df_test = get_adsbx_flights_for_day(val_date)

In [22]:
df_test.sort("takeoff_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a24b39""","""""","""""",2026-03-01 00:00:28.665,"""KDFW""",2026-03-01 00:57:58.725,"""KSGF""",false,false,false,false,"""""","""""","""""",""""""
"""ac69b6""","""""","""""",2026-03-01 00:00:43.583,"""KSAV""",2026-03-01 01:16:58.985,"""KBWI""",false,false,false,false,"""""","""""","""""",""""""
"""abc12a""","""""","""""",2026-03-01 00:01:13.715,"""KPHX""",2026-03-01 02:54:03.958,"""KMDW""",false,false,false,false,"""""","""""","""""",""""""
"""a008c9""","""""","""""",2026-03-01 00:01:28.945,"""KRDU""",2026-03-01 02:32:58.685,"""KMSP""",false,false,false,false,"""""","""""","""""",""""""
"""a0144e""","""""","""""",2026-03-01 00:01:43.720,"""KORD""",2026-03-01 01:44:12.856,"""KLGA""",false,false,false,false,"""""","""""","""""",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ac66d5""","""""","""""",2026-03-01 23:13:12.616,"""KHSV""",2026-03-01 23:46:13.860,"""KATL""",false,false,false,false,"""""","""""","""""",""""""
"""a24369""","""""","""""",2026-03-01 23:18:43.674,"""KLGA""",2026-03-01 23:54:13.823,"""KBOS""",false,false,false,false,"""""","""""","""""",""""""
"""aa6ed1""","""""","""""",2026-03-01 23:19:11.258,"""KDFW""",2026-03-01 23:59:43.865,null,false,false,false,false,"""""","""""","""""",""""""


In [24]:
df_gold.sort("takeoff_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a24b39""",null,"""N247NN""",2026-03-01 00:00:00,"""KDFW""",2026-03-01 00:57:00,"""KSGF""",false,false,false,false,null,null,null,"""A3"""
"""ac69b6""",null,"""N8994Q""",2026-03-01 00:00:00,"""KSAV""",2026-03-01 01:16:00,"""KBWI""",false,false,false,false,null,null,null,"""A3"""
"""a008c9""",null,"""N101DU""",2026-03-01 00:01:00,"""KRDU""",2026-03-01 02:32:00,"""KMSP""",false,false,false,false,null,null,null,"""A3"""
"""a495e9""",null,"""N395FR""",2026-03-01 00:01:00,"""KMCO""",2026-03-01 01:42:00,"""KCVG""",false,false,false,false,null,null,null,"""A3"""
"""abc12a""",null,"""N8568Z""",2026-03-01 00:01:00,"""KPHX""",2026-03-01 02:52:00,"""KMDW""",false,false,false,false,null,null,null,"""A3"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ac66d5""",null,"""N899AT""",2026-03-01 23:13:00,"""KHSV""",2026-03-01 23:46:00,"""KATL""",false,false,false,false,null,null,null,"""A3"""
"""a24369""",null,"""N245JQ""",2026-03-01 23:19:00,"""KLGA""",2026-03-01 23:54:00,"""KBOS""",false,false,false,false,null,null,null,"""A3"""
"""aa6ed1""",null,"""N771XF""",2026-03-01 23:19:00,"""KDFW""",2026-03-02 00:01:00,"""KTUL""",false,false,false,false,null,null,null,"""A3"""


In [25]:
df_test = df_test.filter(pl.col("landing_time").dt.date() == val_date)
df_test = df_test.filter(pl.col("takeoff_time").dt.date() == val_date)
df_test = df_test.filter(pl.col("icao").is_in(df_gold.get_column("icao").unique().implode()))

In [26]:
df, stats = flights_df_comparison(df_test, df_gold)


--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 765
Predicted flights: 3022  |  Gold flights: 3023
Correctly matched: 3022  |  Predicted unmatched: 0  |  Gold flights missed: 1
Precision: 100.00%  |  Recall: 99.97%  |  F1: 99.98%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 3020/3020 matched (100.00%)
landing_airport_ident: 3020/3020 matched (100.00%)
overall: 6040/6040 matched (100.00%)


In [27]:
df.filter(pl.col("match_status") != "matched")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,callsign_df2,registration_df2,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,pia_df2,ladd_df2,military_df2,interesting_df2,aircraft_type_df2,owner_df2,aircraft_description_df2,category_df2,takeoff_time_diff,landing_time_diff,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,duration[ms],duration[ms],str
"""a826f1""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N624UX""",2026-03-01 00:02:00,"""KSLC""",2026-03-01 01:36:00,"""KLAX""",false,false,false,false,null,null,null,"""A3""",null,null,"""df2_only"""


In [28]:
COLUMNS = ["icao","takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "takeoff_time_df2", "landing_time_df2", "takeoff_airport_ident_df2", "landing_airport_ident_df2", "match_status"]

In [19]:
df = df.select(COLUMNS)
df.filter(pl.col("landing_airport_ident") != pl.col("landing_airport_ident_df2"))

icao,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,takeoff_time_df2,landing_time_df2,takeoff_airport_ident_df2,landing_airport_ident_df2,match_status
str,datetime[ms],str,datetime[ms],str,datetime[ms],datetime[ms],str,str,str
"""a6746a""",2026-03-01 19:18:57.922,"""KBFI""",2026-03-01 20:35:25.728,"""KPDX""",2026-03-01 19:19:00,2026-03-01 20:36:00,"""KBFI""","""KVUO""","""matched"""
"""a04db2""",2026-03-01 20:10:13.169,"""KGEU""",2026-03-01 21:18:36.751,"""MMTJ""",2026-03-01 20:10:00,2026-03-01 21:19:00,"""KGEU""","""KSDM""","""matched"""
"""a1990c""",2026-03-01 21:01:28.225,"""KGEG""",2026-03-01 23:18:13.636,"""KPDX""",2026-03-01 21:01:00,2026-03-01 23:19:00,"""KGEG""","""KVUO""","""matched"""
"""a9bd11""",2026-03-01 19:47:28.952,null,2026-03-01 22:36:06.781,"""MMTJ""",2026-03-01 19:47:00,2026-03-01 22:38:00,"""KRHV""","""KSDM""","""matched"""
"""a389c0""",2026-03-01 21:10:43.426,"""KSFO""",2026-03-01 21:52:44.951,"""2Q5""",2026-03-01 21:10:00,2026-03-01 21:51:00,"""KSFO""","""KCXP""","""matched"""
"""a521cd""",2026-03-01 03:45:13.839,"""KSGF""",2026-03-01 05:27:27.939,"""OK67""",2026-03-01 03:45:00,2026-03-01 05:30:00,"""KSGF""","""KDUC""","""matched"""
"""a7bfbb""",2026-03-01 19:42:46.428,"""KBIL""",2026-03-01 21:51:09.131,"""AR13""",2026-03-01 19:47:00,2026-03-01 21:52:00,"""KBIL""","""KCXW""","""matched"""


In [29]:
df.filter(pl.col("icao") =="a826f1")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,callsign_df2,registration_df2,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,pia_df2,ladd_df2,military_df2,interesting_df2,aircraft_type_df2,owner_df2,aircraft_description_df2,category_df2,takeoff_time_diff,landing_time_diff,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,duration[ms],duration[ms],str
"""a826f1""","""""","""""",2026-03-01 03:47:28.789,"""KLAX""",2026-03-01 05:02:13.414,"""KRDD""",false,false,false,false,"""""","""""","""""","""""",null,"""N624UX""",2026-03-01 03:47:00,"""KLAX""",2026-03-01 05:02:00,"""KRDD""",false,false,false,false,null,null,null,"""A3""",28s 789ms,13s 414ms,"""matched"""
"""a826f1""","""""","""""",2026-03-01 15:38:28.730,"""KRDD""",2026-03-01 16:57:28.467,"""KLAX""",false,false,false,false,"""""","""""","""""","""""",null,"""N624UX""",2026-03-01 15:38:00,"""KRDD""",2026-03-01 16:57:00,"""KLAX""",false,false,false,false,null,null,null,"""A3""",28s 730ms,28s 467ms,"""matched"""
"""a826f1""","""""","""""",2026-03-01 18:29:14.001,"""KLAX""",2026-03-01 20:15:58.993,"""KMTJ""",false,false,false,false,"""""","""""","""""","""""",null,"""N624UX""",2026-03-01 18:29:00,"""KLAX""",2026-03-01 20:15:00,"""KMTJ""",false,false,false,false,null,null,null,"""A3""",14s 1ms,58s 993ms,"""matched"""
"""a826f1""","""""","""""",2026-03-01 21:22:13.352,"""KMTJ""",2026-03-01 23:15:13.813,"""KLAX""",false,false,false,false,"""""","""""","""""","""""",null,"""N624UX""",2026-03-01 21:29:00,"""KMTJ""",2026-03-01 23:15:00,"""KLAX""",false,false,false,false,null,null,null,"""A3""",6m 46s 648ms,13s 813ms,"""matched"""
"""a826f1""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N624UX""",2026-03-01 00:02:00,"""KSLC""",2026-03-01 01:36:00,"""KLAX""",false,false,false,false,null,null,null,"""A3""",null,null,"""df2_only"""


In [7]:
df_gold.sort("takeoff_time")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a08df7""","""KAP9""","""N135CA""",2026-03-01 00:00:00,"""KSDY""",2026-03-01 01:26:00,"""KBIL""",false,false,false,false,"""P212""","""HYANNIS AIR SERVICE INC""","""TECNAM P-2012 Traveller""","""A1"""
"""a1a0df""","""N204A""","""N204A""",2026-03-01 00:00:00,"""KPSP""",2026-03-01 00:38:00,"""KVNY""",false,true,false,false,"""GLF5""","""MERCURY INTERNATIONAL HOLDINGS…","""GULFSTREAM 5""","""A3"""
"""a2ff72""","""N2921Q""","""N2921Q""",2026-03-01 00:00:00,"""HAR""",2026-03-01 00:31:00,"""LOM""",false,false,false,false,"""P32R""","""AMBLER AVIATION LLC""","""PIPER PA-32R-300/301/301T""","""A1"""
"""a3a160""","""MVJ33""","""N333EH""",2026-03-01 00:00:00,"""KBZN""",2026-03-01 02:06:00,"""KNUQ""",false,true,false,false,"""GLF5""","""TVPX AIRCRAFT SOLUTIONS INC TR…","""GULFSTREAM 5""","""A3"""
"""a6eed5""","""GJS4497""","""N546GJ""",2026-03-01 00:00:00,"""KORD""",2026-03-01 00:52:00,"""KLEX""",false,false,false,false,"""CRJ7""","""SKYWEST AIRLINES INC""","""BOMBARDIER Regional Jet CRJ-70…","""A2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""a6eed5""","""GJS4573""","""N546GJ""",2026-03-01 23:25:00,"""KMLI""",2026-03-01 23:55:00,"""KORD""",false,false,false,false,"""CRJ7""","""SKYWEST AIRLINES INC""","""BOMBARDIER Regional Jet CRJ-70…","""A2"""
"""ad8c37""","""N972MS""","""N972MS""",2026-03-01 23:27:00,"""KPBI""",2026-03-01 23:48:00,"""KFXE""",false,true,false,false,"""GLF4""","""GLF4 1285 LLC""","""GULFSTREAM 4""","""A2"""
"""a6d6ec""","""N54PV""","""N54PV""",2026-03-01 23:28:00,"""KLGB""",2026-03-01 23:50:00,"""KCMA""",false,false,false,false,"""C510""","""NOSTROMO 2021 LLC""","""CESSNA 510 Citation Mustang""","""A1"""


In [15]:
df.filter(pl.col("match_status") == "df1_only")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,callsign_df2,registration_df2,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,pia_df2,ladd_df2,military_df2,interesting_df2,aircraft_type_df2,owner_df2,aircraft_description_df2,category_df2,takeoff_time_diff,landing_time_diff,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,duration[ms],duration[ms],str
"""046c56""","""""","""""",2026-03-01 13:03:38.420,"""""",2026-03-01 13:52:33.460,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 14:32:46.860,"""""",2026-03-01 16:24:12.920,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 18:41:27.650,"""""",2026-03-01 19:59:43.450,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 20:43:21.070,"""""",2026-03-01 21:00:47.270,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""046c56""","""""","""""",2026-03-01 21:38:18.270,"""""",2026-03-01 22:11:18.120,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""add7dc""","""""","""""",2026-03-01 00:52:21.670,"""""",2026-03-01 01:23:45.780,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""ade27a""","""""","""""",2026-03-01 17:21:18.040,"""""",2026-03-01 18:35:55.530,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""
"""ade27a""","""""","""""",2026-03-01 19:42:52.710,"""""",2026-03-01 20:18:26.480,"""""",false,false,false,false,"""""","""""","""""","""""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""df1_only"""


In [13]:
df.filter(pl.col("icao") == "c06401")

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,callsign_df2,registration_df2,takeoff_time_df2,takeoff_airport_ident_df2,landing_time_df2,landing_airport_ident_df2,pia_df2,ladd_df2,military_df2,interesting_df2,aircraft_type_df2,owner_df2,aircraft_description_df2,category_df2,takeoff_time_diff,landing_time_diff,match_status
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str,duration[ms],duration[ms],str
"""c06401""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""CGLWQ""","""C-GLWQ""",2026-03-01 19:49:00,"""KMKE""",2026-03-01 20:12:00,"""KGRR""",false,false,false,false,"""P180""","""Flightpath Charter Airways Inc…","""PIAGGIO P-180 Avanti""","""A2""",null,null,"""df2_only"""


In [14]:
df_2 = read_icaos_from_adsb(date(2026,3,1), icaos=["c06401"])
df_2

time,icao,registration,callsign,lat,lon,baro_altitude_ft,on_ground,ground_speed_kt,track_deg,aircraft_type,aircraft_description,owner,category,pia,ladd,military,interesting
datetime[ms],str,str,str,f64,f64,i32,bool,f32,f32,str,str,str,str,bool,bool,bool,bool
2026-03-01 19:49:18.360,"""c06401""","""C-GLWQ""","""CGLWQ""",42.943695,-87.895333,425,false,152.300003,7.5,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:19.370,"""c06401""","""C-GLWQ""",null,42.944458,-87.89516,500,false,152.300003,7.5,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:19.860,"""c06401""","""C-GLWQ""",null,42.944824,-87.895096,525,false,153.399994,7.9,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:20.670,"""c06401""","""C-GLWQ""",null,42.945419,-87.894968,600,false,154.600006,8.2,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 19:49:22.290,"""c06401""","""C-GLWQ""","""CGLWQ""",42.946518,-87.894777,700,false,155.600006,8.1,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-03-01 20:07:50.360,"""c06401""","""C-GLWQ""",null,42.875216,-85.555115,500,false,118.199997,81.699997,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 20:07:50.910,"""c06401""","""C-GLWQ""",null,42.875244,-85.554753,500,false,118.199997,81.699997,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…",null,false,false,false,false
2026-03-01 20:07:51.820,"""c06401""","""C-GLWQ""","""CGLWQ""",42.87529,-85.55405,475,false,118.199997,81.699997,"""P180""","""PIAGGIO P-180 Avanti""","""Flightpath Charter Airways Inc…","""A2""",false,false,false,false


In [1]:
from datetime import date
import polars as pl

from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from flights_algorithm.main import run_main_2
from flights_ai_model.validation import flights_df_comparison_by_validation_tier

target_date = date(2026, 3, 1)

# 1. Your ADS-B-derived algorithm output
df_test = run_main_2(target_date)

# 2. ADSB Exchange flights-v2 arrivals for the day
df_gold = get_adsbx_flights_for_day(target_date)

# ADS-B messages used only to label gold rows as adsb_verified vs hard/unverified
df_adsb = read_adsb(target_date)

# Important: ADSBX file is arrivals-by-day, while run_main_2 is takeoffs-by-day.
# For an arrivals eval, compare only flights landing on target_date.
df_test = df_test.filter(pl.col("landing_time").dt.date() == target_date)
df_gold = df_gold.filter(pl.col("landing_time").dt.date() == target_date)

outputs = flights_df_comparison_by_validation_tier(df_test, df_gold, df_adsb)

df_all, stats_all = outputs["all"]
df_verified, stats_verified = outputs["adsb_verified"]
df_hard, stats_hard = outputs["unverified_or_hard"]

stats_all, stats_verified, stats_hard

KeyboardInterrupt: 

In [4]:
from datetime import date
import polars as pl

from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.adsbx.get_flights import get_adsbx_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from flights_algorithm.main import run_main
from flights_ai_model.validation import flights_df_comparison_by_validation_tier

target_date = date(2026, 3, 1)

df_adsb = read_adsb(target_date)

df_gold = get_sfdps_flights_day(target_date)

df_run_main_2 = run_main(target_date)
df_adsbx = get_adsbx_flights_for_day(target_date)

# Match SFDPS' same-day flight definition.
same_day_expr = (
    (pl.col("takeoff_time").dt.date() == target_date) &
    (pl.col("landing_time").dt.date() == target_date)
)

df_gold = df_gold.filter(same_day_expr)
df_run_main_2 = df_run_main_2.filter(same_day_expr)
df_adsbx = df_adsbx.filter(same_day_expr)

run_main_2_outputs = flights_df_comparison_by_validation_tier(
    df_run_main_2,
    df_gold,
    df_adsb,
)

adsbx_outputs = flights_df_comparison_by_validation_tier(
    df_adsbx,
    df_gold,
    df_adsb,
)

run_main_2_stats = {
    tier: stats
    for tier, (_df, stats) in run_main_2_outputs.items()
}

adsbx_stats = {
    tier: stats
    for tier, (_df, stats) in adsbx_outputs.items()
}

run_main_2_stats, adsbx_stats


--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 12308
Predicted flights: 32689  |  Gold flights: 27082
Correctly matched: 24903  |  Predicted unmatched: 7786  |  Gold flights missed: 2179
Precision: 76.18%  |  Recall: 91.95%  |  F1: 83.33%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 22408/23379 matched (95.85%)
landing_airport_ident: 22773/23218 matched (98.08%)
overall: 45181/46597 matched (96.96%)

--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 3484
Predicted flights: 7329  |  Gold flights: 7449
Correctly matched: 7091  |  Predicted unmatched: 238  |  Gold flights missed: 358
Precision: 96.75%  |  Recall: 95.19%  |  F1: 95.97%

--- Airport Ident Agreement (on time-matched flights) ---
takeoff_airport_ident: 6544/6689 matched (97.83%)
landing_airport_ident: 6529/6625 matched (98.55%)
overall: 13073/13314 matched (98.19%)

--- Match Summary (ICAO-based, ±30 min window) ---
Common ICAOs: 8824
Predicted flights: 25

({'all': {'n_common_icaos': 12308,
   'n_predicted': 32689,
   'n_gold': 27082,
   'n_correctly_matched': 24903,
   'n_predicted_unmatched': 7786,
   'n_gold_missed': 2179,
   'precision': 0.7618159013735507,
   'recall': 0.9195406543091352,
   'f1': 0.8332803533486138,
   'airport_ident_pct_match': 0.9696117775822478},
  'adsb_verified': {'n_common_icaos': 3484,
   'n_predicted': 7329,
   'n_gold': 7449,
   'n_correctly_matched': 7091,
   'n_predicted_unmatched': 238,
   'n_gold_missed': 358,
   'precision': 0.9675262655205349,
   'recall': 0.95193985769902,
   'f1': 0.9596697794018135,
   'airport_ident_pct_match': 0.9818987531921286},
  'unverified_or_hard': {'n_common_icaos': 8824,
   'n_predicted': 25360,
   'n_gold': 19633,
   'n_correctly_matched': 17812,
   'n_predicted_unmatched': 7548,
   'n_gold_missed': 1821,
   'precision': 0.7023659305993691,
   'recall': 0.9072480008149544,
   'f1': 0.7917676082946237,
   'airport_ident_pct_match': 0.9646966920049275}},
 {'all': {'n_comm

In [5]:
run_main_2_stats

{'all': {'n_common_icaos': 12308,
  'n_predicted': 32689,
  'n_gold': 27082,
  'n_correctly_matched': 24903,
  'n_predicted_unmatched': 7786,
  'n_gold_missed': 2179,
  'precision': 0.7618159013735507,
  'recall': 0.9195406543091352,
  'f1': 0.8332803533486138,
  'airport_ident_pct_match': 0.9696117775822478},
 'adsb_verified': {'n_common_icaos': 3484,
  'n_predicted': 7329,
  'n_gold': 7449,
  'n_correctly_matched': 7091,
  'n_predicted_unmatched': 238,
  'n_gold_missed': 358,
  'precision': 0.9675262655205349,
  'recall': 0.95193985769902,
  'f1': 0.9596697794018135,
  'airport_ident_pct_match': 0.9818987531921286},
 'unverified_or_hard': {'n_common_icaos': 8824,
  'n_predicted': 25360,
  'n_gold': 19633,
  'n_correctly_matched': 17812,
  'n_predicted_unmatched': 7548,
  'n_gold_missed': 1821,
  'precision': 0.7023659305993691,
  'recall': 0.9072480008149544,
  'f1': 0.7917676082946237,
  'airport_ident_pct_match': 0.9646966920049275}}

In [6]:
adsbx_stats

{'all': {'n_common_icaos': 11369,
  'n_predicted': 29890,
  'n_gold': 27082,
  'n_correctly_matched': 24099,
  'n_predicted_unmatched': 5791,
  'n_gold_missed': 2983,
  'precision': 0.8062562730010037,
  'recall': 0.889853038918839,
  'f1': 0.8459945236256406,
  'airport_ident_pct_match': 0.9872193866965434},
 'adsb_verified': {'n_common_icaos': 3217,
  'n_predicted': 6948,
  'n_gold': 7449,
  'n_correctly_matched': 6812,
  'n_predicted_unmatched': 136,
  'n_gold_missed': 637,
  'precision': 0.9804260218767991,
  'recall': 0.9144851657940664,
  'f1': 0.9463082586649996,
  'airport_ident_pct_match': 0.9983118027011156},
 'unverified_or_hard': {'n_common_icaos': 8152,
  'n_predicted': 22942,
  'n_gold': 19633,
  'n_correctly_matched': 17287,
  'n_predicted_unmatched': 5655,
  'n_gold_missed': 2346,
  'precision': 0.7535088484003138,
  'recall': 0.8805073091223959,
  'f1': 0.8120728126834996,
  'airport_ident_pct_match': 0.9828483831781107}}